<a href="https://colab.research.google.com/github/sadot04/Inteligencia_artificial/blob/main/Ejercicios_deep_Learning_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [ ]:
#que no memorice a rajatabla, si no que los entienda
SEED = 42
os.environ["PYTHOMHSSHSEED"] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("Tensorflow:", tf.__version__)

Tensorflow: 2.19.0


In [ ]:
dataset = pd.read_csv("Churn_Modelling.csv")
X = dataset.iloc[:, 3:-1].copy() #X es variable independiente
y = dataset.iloc[:, -1].values  #y var dependiente
print("Shapes:", X.shape, y.shape)

Shapes: (10000, 10) (10000,)


In [ ]:
le_gender = LabelEncoder()
X.loc[:,  X.columns[2]] = le_gender.fit_transform(X.iloc[:, 2])
ct = ColumnTransformer(
    transformers=[('geo_ohe', OneHotEncoder(handle_unknown="ignore"), [1])],
                       remainder='passthrough')
X_ohe = ct.fit_transform(X)
X_ohe = np.asarray(X_ohe).astype('float32')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_ohe, y, test_size=0.2, random_state=SEED, stratify=y
    )

sc = StandardScaler(with_mean=False)#with_mean se usa cuando los datos pueden ser dispersos
X_train = sc.fit_transform(X_train).astype('float32')
X_test = sc.transform(X_test).astype('float32')

In [ ]:
#Aqui entra la red neuronal artificial
from tensorflow.keras import layers, models, callbacks
def build_model(input_dim):
    model = models.Sequential([
        layers.Dense(16, activation='relu', input_shape=(input_dim,)),
        layers.Dense(16, activation='relu'),
        layers.Dense(1, activation='sigmoid')#sigmoid para las salidas 0 y 1
    ])
    model.compile(
        optimizer='adam', #optimizador adam es para multiples caracteristicas de valores de entrada
        loss='binary_crossentropy', #entropia cruzada, verificamos valor de perdida, binaria porque es 0 y 1
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model
ann = build_model(input_dim=X_train.shape[1])
ann.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 16)             │           208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 497 (1.94 KB)

 Trainable params: 497 (1.94 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
cb = [
    callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor = "val_auc", mode="max"), #El callback.Early, es para entrenamiento, no mejora hasta 10 epocas
    callbacks.ModelCheckpoint("best_ann.keras", monitor="val_auc", mode="max", save_best_only=True), #guarda el mejor de los resultados
    callbacks.ReduceLROnPlateau(monitor="val_loss", mode="max", patience=5, factor=0.5)
]

In [ ]:
hist = ann.fit(
    X_train, y_train, #datos de entrenamient
    validation_split=0.2, #validacion
    epochs=200, #200 iteraciones epocas
    batch_size=32, #en lugar de entrenar con todo, se entrena con subconjuntos de 32 ejemplos por lote
    callbacks=cb,
    verbose=1
)

Epoch 1/200
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.7122 - auc: 0.5432 - loss: 0.5898 - val_accuracy: 0.7994 - val_auc: 0.6783 - val_loss: 0.4696 - learning_rate: 0.0010
Epoch 2/200
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7915 - auc: 0.7022 - loss: 0.4718 - val_accuracy: 0.8056 - val_auc: 0.7376 - val_loss: 0.4409 - learning_rate: 0.0010
Epoch 3/200
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7987 - auc: 0.7512 - loss: 0.4460 - val_accuracy: 0.8138 - val_auc: 0.7492 - val_loss: 0.4331 - learning_rate: 0.0010
Epoch 4/200
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8048 - auc: 0.7651 - loss: 0.4354 - val_accuracy: 0.8213 - val_auc: 0.7563 - val_loss: 0.4282 - learning_rate: 0.0010
Epoch 5/200
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8110 - auc: 0.7745 - loss: 0.4285 - val_accuracy: 0.8238 - val_auc: 0.7651 - val_loss: 0.4221 - learning_rate: 0.0010
Epoch 6/200
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8132 - a

In [ ]:
y_proba = ann.predict(X_test)
y_pred = (y_proba > 0.5).astype(int)

print("\nMatriz de confusion:\n", confusion_matrix(y_test, y_pred))
print("\nReporte de clasificacion:\n", classification_report(y_test, y_pred))
print("\nAUC: ", roc_auc_score(y_test, y_proba))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

Matriz de confusion:
 [[1546   47]
 [ 255  152]]

Reporte de clasificacion:
               precision    recall  f1-score   support

           0       0.86      0.97      0.91      1593
           1       0.76      0.37      0.50       407

    accuracy                           0.85      2000
   macro avg       0.81      0.67      0.71      2000
weighted avg       0.84      0.85      0.83      2000


AUC:  0.8320678151186626


In [ ]:
sample_raw = pd.DataFrame(  #Datos al azar para ver que responde el modelo en la ann
    [{
        "CreditScore": 600,
        "Geography": "France",
        "Gender": "Male",
        "Age": 40,
        "Tenure": 3,
        "Balance": 60000,
        "NumOfProducts": 2,
        "HasCrCard": 1,
        "IsActiveMember": 1,
        "EstimatedSalary": 50000
    }])
sample_raw.loc[:, "Gender"] = le_gender.transform(sample_raw.loc[:, "Gender"])
sample_ohe = ct.transform(sample_raw)
sample_scaled = sc.transform(sample_ohe).astype('float32')

proba = ann.predict(sample_scaled).item()
print(f"\nProbablididad de churn para el cliente ejemplo: {proba:.4f}")
print("Salir del banco" if proba >= 0.5 else "Permaneces en el banco")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step

Probablididad de churn para el cliente ejemplo: 0.0750
Permaneces en el banco
